# Massive flat files: NQ minute bars

**Goal.** Build an NQ minute series from 2017 through 2026 so the hourly pattern can be tested outside the Yahoo window.

**Method.** Massive Flat Files, bucket `flatfiles`, prefix `us_futures_cme/minute_aggs_v1/`. The client in `src.massive` reads `MASSIVE_ACCESS_KEY` and `MASSIVE_SECRET_KEY` from `.env`. Daily files are filtered to outright NQ contracts (`NQ` plus a quarter code `H/M/U/Z` and one digit). `src.front_month` keeps, each session, the outright with the highest volume and converts timestamps to America/New_York.

**Result.** First download on Nov 25, 2026: 3,384,521 front-month minute bars, 2017-04-02 18:00 through 2026-11-24 16:59 New York, 39 contracts from NQM7 to NQZ6. Prices are not back-adjusted. Do not compute a return across a roll. The parquet files are licensed and are listed in `.gitignore`. If the year files are already in `data/`, the download cell skips them.


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


In [1]:
from src.massive import s3_client, list_child_prefixes

s3 = s3_client()
top_prefixes = list_child_prefixes(s3)
print("Datasets:")
for prefix in top_prefixes:
    print(" ", prefix)

for prefix in top_prefixes:
    if "fut" in prefix.lower():
        print(f"\nInside {prefix}:")
        for child in list_child_prefixes(s3, prefix):
            print("  ", child)


Datasets:
  global_crypto/
  global_forex/
  us_futures_cbot/
  us_futures_cme/
  us_futures_comex/
  us_futures_nymex/
  us_indices/
  us_options_opra/
  us_stocks_sip/

Inside us_futures_cbot/:
   us_futures_cbot/minute_aggs_v1/
   us_futures_cbot/quotes_v1/
   us_futures_cbot/session_aggs_v1/
   us_futures_cbot/trades_v1/

Inside us_futures_cme/:
   us_futures_cme/minute_aggs_v1/
   us_futures_cme/quotes_v1/
   us_futures_cme/session_aggs_v1/
   us_futures_cme/trades_v1/

Inside us_futures_comex/:
   us_futures_comex/minute_aggs_v1/
   us_futures_comex/quotes_v1/
   us_futures_comex/session_aggs_v1/
   us_futures_comex/trades_v1/

Inside us_futures_nymex/:
   us_futures_nymex/minute_aggs_v1/
   us_futures_nymex/quotes_v1/
   us_futures_nymex/session_aggs_v1/
   us_futures_nymex/trades_v1/


In [2]:
import io
import pandas as pd

from src.massive import BUCKET, MINUTE_PREFIX, list_child_prefixes, list_keys

prefix = MINUTE_PREFIX
years = [item.split("/")[-2] for item in list_child_prefixes(s3, prefix)]
print("Years available:", years)

months = list_child_prefixes(s3, f"{prefix}{years[-1]}/")
month = months[0]
keys = list_keys(s3, month)
print(f"\n{len(keys)} files in {month}, e.g. {keys[0]}")

obj = s3.get_object(Bucket=BUCKET, Key=keys[0])
sample = pd.read_csv(io.BytesIO(obj["Body"].read()), compression="gzip")
print(f"\nShape: {sample.shape}")
print("Columns:", sample.columns.tolist())
print(sample.head())

tcol = [c for c in sample.columns if "tick" in c.lower() or "sym" in c.lower()][0]
nq_rows = sample[sample[tcol].astype(str).str.startswith("NQ")]
print(f"\nNQ tickers in this file ({tcol} column):")
print(nq_rows[tcol].value_counts().head(10))


Years available: ['2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026']

21 files in us_futures_cme/minute_aggs_v1/2026/01/, e.g. us_futures_cme/minute_aggs_v1/2026/01/2026-01-02.csv.gz

Shape: (64768, 11)
Columns: ['ticker', 'exchange', 'session_end_date', 'window_start', 'open', 'high', 'low', 'close', 'volume', 'dollar_volume', 'transactions']
  ticker  exchange session_end_date         window_start      open      high  \
0   6AH6         4       2026-01-02  1767308400000000000  0.667800  0.667800   
1   6BH6         4       2026-01-02  1767308400000000000  1.347200  1.347400   
2   6CH6         4       2026-01-02  1767308400000000000  0.730950  0.731200   
3   6EH6         4       2026-01-02  1767308400000000000  1.178600  1.178800   
4   6JH6         4       2026-01-02  1767308400000000000  0.006422  0.006422   

        low     close  volume  dollar_volume  transactions  
0  0.666500  0.667250     320     213.542200            87  
1  1.347100  1.347400 

In [4]:
from src.massive import download_nq_minutes

# Existing year files are skipped. The stored output is the first download.
download_nq_minutes(s3, years)


2017: 189 days, 318,317 NQ rows saved
2018: 252 days, 447,778 NQ rows saved
2019: 254 days, 431,539 NQ rows saved
2020: 261 days, 435,936 NQ rows saved
2021: 260 days, 453,137 NQ rows saved
2022: 260 days, 458,875 NQ rows saved
2023: 258 days, 456,317 NQ rows saved
2024: 258 days, 466,722 NQ rows saved
2025: 256 days, 456,447 NQ rows saved
2026: 234 days, 426,229 NQ rows saved


In [5]:
from src.front_month import build_front_month

year_files = [ROOT / "data" / f"nq_1m_{year}.parquet" for year in years]
nq = build_front_month(year_files)
print(f"Bars: {len(nq):,} | {nq.index[0]} to {nq.index[-1]}")
print("Contracts used:", nq["ticker"].drop_duplicates().tolist())


Bars: 3,384,521 | 2017-04-02 18:00:00-04:00 to 2026-11-24 16:59:00-05:00
Contracts used: ['NQM7', 'NQU7', 'NQZ7', 'NQH8', 'NQM8', 'NQU8', 'NQZ8', 'NQH9', 'NQM9', 'NQU9', 'NQZ9', 'NQH0', 'NQM0', 'NQU0', 'NQZ0', 'NQH1', 'NQM1', 'NQU1', 'NQZ1', 'NQH2', 'NQM2', 'NQU2', 'NQZ2', 'NQH3', 'NQM3', 'NQU3', 'NQZ3', 'NQH4', 'NQM4', 'NQU4', 'NQZ4', 'NQH5', 'NQM5', 'NQU5', 'NQZ5', 'NQH6', 'NQM6', 'NQU6', 'NQZ6']
